# Stage 1 Model Training for Predicting Peak Drift of Circular RC Columns


In [ ]:
import os, time, json, warnings, functools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import shap
from joblib import Parallel, delayed, parallel_backend

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR as SkSVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import xgboost as xgb
import catboost as cat
import lightgbm as lgb

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')


## Configuration

In [ ]:
QUICK_TEST = False   # True: fast environment check (6,000-row sample, 2 trials; results are
                     # NOT meaningful and are written to output_quicktest/ instead of output/)

# Paths are relative to the folder this notebook is run from, so the folder
# (notebook + ml_stage1_db.csv) can be copied anywhere and run as is.
HERE = os.getcwd()
CSV_PATH = next((p for p in (os.path.join(HERE, 'ml_stage1_db.csv'),
                             os.path.join(HERE, '..', 'ml_stage1_db.csv'))
                 if os.path.exists(p)), None)
assert CSV_PATH is not None, f'ml_stage1_db.csv not found next to the notebook ({HERE})'
OUT_DIR  = os.path.join(HERE, 'output_quicktest' if QUICK_TEST else 'output')   # all result files go here
SHAP_DIR = os.path.join(OUT_DIR, 'shap')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(SHAP_DIR, exist_ok=True)

RNG_SEED      = 29     # final chosen seed (CatBoost wins under full GroupKFold HPO;
                       # feature selection independently reproduces the 5-variable set)
TEST_RATIO    = 0.30
N_CV_HPO      = 10     # GroupKFold splits
N_JOBS        = min(10, max(1, (os.cpu_count() or 2) - 1))  # parallel workers inside each model
                       # (CV folds / tree threads); 1 = fully sequential. Capped at the 10 CV folds.
N_JOBS_SHAP   = max(1, (os.cpu_count() or 2) - 1)           # worker processes for the RF / SVR SHAP
N_TRIALS      = 50     # Optuna trials per model
SUBSAMPLE_SVR = 20_000  # SVR RBF is O(n^2); HPO+fit on a subsample like the production script
SHAP_ROWS_SLOW = 2_000  # Random Forest / SVR SHAP on this many random rows (None = all 189,572 rows).
                        # Single-process, the SVR KernelExplainer costs ~2.5 s/row (days for all rows);
                        # the full-data RF/SVR values come from 03_shap_rf_svr.py (30 parallel workers).
                        # Tree models (DT, XGBoost, LightGBM, CatBoost) always use all rows.
if QUICK_TEST:   # tiny settings: only checks that the environment and every cell run
    N_TRIALS, SUBSAMPLE_SVR, SHAP_ROWS_SLOW = 2, 1_500, 200
    N_JOBS_SHAP = min(N_JOBS_SHAP, 4)

MODEL_COLORS = {
    'Decision Tree': '#6491F2', 'Random Forest': '#40D9D4', 'SVR': '#FFD688',
    'XGBoost': '#FF8C8C', 'LightGBM': '#66DDB3', 'CatBoost': '#F462B9',
}

fonttitle = {'fontname': 'Times New Roman', 'fontsize': 12, 'fontweight': 'bold'}
fontdict  = {'fontname': 'Times New Roman', 'fontsize': 10, 'fontweight': 'bold'}


## Data Loading and Exploration

In [ ]:
data = pd.read_csv(CSV_PATH)
if QUICK_TEST:
    data = data.sample(6000, random_state=0).reset_index(drop=True)
print(data.shape)
data.info()
data.head()


In [ ]:
FEAT_DEFS = [
    ('D_mm',              'log_D'),
    ('LD',                'log_LD'),
    ('rhoS',              'log_rhoS'),
    ('PGA_g',             'log_PGA'),
    ('theta_r_final_abs', 'log_theta_r'),
]
feat_names = [d[1] for d in FEAT_DEFS]
feat_disp  = ['D', 'L/D', 'rhoS', 'PGA', 'theta_r']

X_raw = np.column_stack([np.log(data[col].values.astype(float)) for col, _ in FEAT_DEFS])
y_raw = np.log(data['theta_max_abs'].values.astype(float))

valid = np.all(np.isfinite(X_raw), axis=1) & np.isfinite(y_raw)
X = X_raw[valid]
y = y_raw[valid]
model_ids  = data['ModelID'].values[valid]
theta_r_sv = data['theta_r_final_abs'].values[valid]
n = len(y)
print(f'Final: {n:,} rows x {len(FEAT_DEFS)} features')


## Train/Test Split (ModelID-level, grouped, 70:30)


In [ ]:
unique_mids = np.unique(model_ids)
n_models    = len(unique_mids)
n_rows_per  = np.array([(model_ids == mid).sum() for mid in unique_mids])
target_test = round(n * TEST_RATIO)

rng  = np.random.default_rng(RNG_SEED)
perm = rng.permutation(n_models)
cum  = np.cumsum(n_rows_per[perm])
k    = int(np.argmin(np.abs(cum - target_test)))
k    = max(1, min(n_models - 1, k))
mid_te = unique_mids[perm[:k + 1]]
mid_tr = unique_mids[perm[k + 1:]]
idx_te = np.where(np.isin(model_ids, mid_te))[0]
idx_tr = np.where(np.isin(model_ids, mid_tr))[0]

X_train, y_train = X[idx_tr], y[idx_tr]
X_test,  y_test  = X[idx_te], y[idx_te]
X_all = X
groups_train = model_ids[idx_tr]

print(f'Train: {len(mid_tr)} models, {len(idx_tr):,} rows')
print(f'Test : {len(mid_te)} models, {len(idx_te):,} rows ({len(idx_te)/n:.1%})')

split_info = pd.DataFrame({
    'ModelID': np.concatenate([mid_tr, mid_te]),
    'set': ['train'] * len(mid_tr) + ['test'] * len(mid_te),
    'n_rows': np.concatenate([
        [n_rows_per[np.where(unique_mids == m)[0][0]] for m in mid_tr],
        [n_rows_per[np.where(unique_mids == m)[0][0]] for m in mid_te],
    ]),
})
split_info.to_csv(os.path.join(OUT_DIR, 'split_info.csv'), index=False)


## Evaluation Results Table


In [ ]:
evaluation_reg = pd.DataFrame({
    'Model': [], 'Details': [],
    'RMSE (train)': [], 'R-squared (train)': [],
    'RMSE (test)': [], 'R-squared (test)': [],
    '10-Fold CV': [], 'Elapsed time': [],
})

cv_hpo = GroupKFold(n_splits=N_CV_HPO)
predictions_all = []   # row-level predictions, all models, train+test
trial_rows_all  = []   # HPO convergence history, all models
best_params_all = {}   # per-model best hyperparameters
shap_vals_all   = {}   # per-model SHAP values (applied to X_all)


def _shap_chunk(kind, model, bg, Xc):
    # SHAP values for one block of rows (runs in a worker process when parallel).
    # SVR: KernelExplainer enumerates all 2^5 coalitions exactly (nsamples >= 2^5 - 2).
    if kind == 'kernel':
        return shap.KernelExplainer(model.predict, bg).shap_values(Xc, nsamples=512, silent=True)
    return shap.TreeExplainer(model).shap_values(Xc)


def train_evaluate_model(model_name, objective_fn, build_fn,
                          Xtr_fit, ytr_fit, groups_fit, details='All features', fold_jobs=1):
    """HPO (Optuna, GroupKFold) -> final fit -> evaluate -> log -> plot -> SHAP."""
    t0 = time.perf_counter()

    trial_rows = []
    best_so_far = [-np.inf]

    def _objective(trial):
        m = objective_fn(trial)
        if fold_jobs > 1:   # CV folds in parallel threads (the native fits release the GIL)
            with parallel_backend('threading', n_jobs=fold_jobs):
                scores = cross_val_score(m, Xtr_fit, ytr_fit, cv=cv_hpo, groups=groups_fit,
                                          scoring='r2', n_jobs=fold_jobs)
        else:
            scores = cross_val_score(m, Xtr_fit, ytr_fit, cv=cv_hpo, groups=groups_fit,
                                      scoring='r2', n_jobs=1)
        val = float(scores.mean())
        best_so_far[0] = max(best_so_far[0], val)
        trial_rows.append({'model': model_name, 'trial': trial.number + 1,
                            'value': val, 'best_so_far': best_so_far[0]})
        return val

    sampler = optuna.samplers.TPESampler(seed=RNG_SEED)
    study = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(_objective, n_trials=N_TRIALS, show_progress_bar=False)
    best_params = dict(study.best_params)
    cv_r2 = study.best_value

    model = build_fn(best_params)
    model.fit(Xtr_fit, ytr_fit)

    pred_tr = np.asarray(model.predict(X_train)).ravel()   # full training set (not SVR subsample)
    pred_te = np.asarray(model.predict(X_test)).ravel()
    rmse_tr = float(np.sqrt(mean_squared_error(y_train, pred_tr)))
    r2_tr   = float(r2_score(y_train, pred_tr))
    rmse_te = float(np.sqrt(mean_squared_error(y_test, pred_te)))
    r2_te   = float(r2_score(y_test, pred_te))
    elapsed = time.perf_counter() - t0

    r = evaluation_reg.shape[0]
    evaluation_reg.loc[r] = [model_name, details, rmse_tr, r2_tr, rmse_te, r2_te, cv_r2, elapsed]
    print(f'{model_name}: RMSE(te)={rmse_te:.4f}  R2(te)={r2_te:.4f}  '
          f'CV_R2={cv_r2:.4f}  elapsed={elapsed:.1f}s')

    for set_name, idx_set, y_true_set, y_pred_set in [
        ('train', idx_tr, y_train, pred_tr), ('test', idx_te, y_test, pred_te)]:
        mids_set = model_ids[idx_set]
        thr_set  = theta_r_sv[idx_set]
        for i in range(len(y_true_set)):
            predictions_all.append({
                'model': model_name, 'set': set_name, 'ModelID': mids_set[i],
                'theta_r_final': float(thr_set[i]),
                'y_true': float(y_true_set[i]), 'y_pred': float(y_pred_set[i]),
                'prediction_error': float(y_pred_set[i] - y_true_set[i]),
            })
    trial_rows_all.extend(trial_rows)
    best_params_all[model_name] = best_params

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    for ax, (set_lbl, yt, yp, r2v) in zip(
            axes, [('Train', y_train, pred_tr, r2_tr), ('Test', y_test, pred_te, r2_te)]):
        ax.plot(yt, yt, color='r', label=f'$R^2$ = {r2v:.2f}')
        ax.scatter(yt, yp, c=MODEL_COLORS[model_name], edgecolors='k', linewidths=0.3, s=10,
                   alpha=0.5, zorder=1)
        ax.set_xlabel('Actual data', **fontdict)
        ax.set_ylabel('Predicted data', **fontdict)
        ax.set_title(set_lbl, **fonttitle)
        ax.legend()
    fig.suptitle(model_name, **fonttitle)
    fig.tight_layout()
    plt.show()

    try:
        if (model_name in ('Random Forest', 'SVR') and SHAP_ROWS_SLOW is not None
                and SHAP_ROWS_SLOW < len(X_all)):
            rng_shap = np.random.default_rng(RNG_SEED + 101)
            X_shap = X_all[rng_shap.choice(len(X_all), size=SHAP_ROWS_SLOW, replace=False)]
        else:
            X_shap = X_all
        bg = shap.kmeans(Xtr_fit, min(100, len(Xtr_fit))) if model_name == 'SVR' else None
        kind = 'kernel' if model_name == 'SVR' else 'tree'
        if model_name in ('Random Forest', 'SVR') and N_JOBS_SHAP > 1 and len(X_shap) >= 2 * N_JOBS_SHAP:
            parts = Parallel(n_jobs=N_JOBS_SHAP)(
                delayed(_shap_chunk)(kind, model, bg, c) for c in np.array_split(X_shap, N_JOBS_SHAP))
            sv = np.vstack([np.asarray(p[0] if isinstance(p, list) else p) for p in parts])
        else:
            sv = _shap_chunk(kind, model, bg, X_shap)
        if isinstance(sv, list):
            sv = sv[0]
        if np.asarray(sv).ndim == 3:
            sv = np.asarray(sv)[:, :, 0]
        shap_vals_all[model_name] = np.asarray(sv)
    except Exception as e:
        print(f'  SHAP warning [{model_name}]: {e}')

    return model, best_params


## Decision Tree

In [ ]:
def _dt_objective(trial):
    p = dict(
        max_depth=trial.suggest_int('max_depth', 2, 25),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 20),
    )
    return DecisionTreeRegressor(random_state=0, **p)

def _dt_build(params):
    return DecisionTreeRegressor(random_state=0, **params)

dt_model, dt_params = train_evaluate_model(
    'Decision Tree', _dt_objective, _dt_build, X_train, y_train, groups_train)


## Random Forest

In [ ]:
def _rf_objective(trial):
    p = dict(
        n_estimators=trial.suggest_int('n_estimators', 50, 600),
        max_depth=trial.suggest_int('max_depth', 3, 30),
        min_samples_split=trial.suggest_int('min_samples_split', 2, 20),
        min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 20),
        max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2']),
    )
    return RandomForestRegressor(random_state=0, n_jobs=N_JOBS, **p)   # tree-level threads (memory-safe)

def _rf_build(params):
    return RandomForestRegressor(random_state=0, n_jobs=N_JOBS, **params)

rf_model, rf_params = train_evaluate_model(
    'Random Forest', _rf_objective, _rf_build, X_train, y_train, groups_train)


## SVR

In [ ]:
rng_svr = np.random.default_rng(RNG_SEED + 13)
sub_svr = rng_svr.choice(len(X_train), size=min(SUBSAMPLE_SVR, len(X_train)), replace=False)
X_train_svr = X_train[sub_svr]
y_train_svr = y_train[sub_svr]
groups_svr  = groups_train[sub_svr]

def _svr_objective(trial):
    gamma_type = trial.suggest_categorical('gamma_type', ['scale', 'float'])
    gamma = ('scale' if gamma_type == 'scale'
             else trial.suggest_float('gamma_float', 1e-3, 10.0, log=True))
    p = dict(C=trial.suggest_float('C', 0.01, 200.0, log=True),
              epsilon=trial.suggest_float('epsilon', 0.001, 1.0, log=True), gamma=gamma)
    return Pipeline([('scaler', StandardScaler()), ('svr', SkSVR(kernel='rbf', **p))])

def _svr_build(params):
    p2 = dict(params)
    gamma_type = p2.pop('gamma_type', None)
    gamma_float = p2.pop('gamma_float', None)
    gamma = 'scale' if gamma_type == 'scale' else gamma_float
    return Pipeline([('scaler', StandardScaler()), ('svr', SkSVR(kernel='rbf', gamma=gamma, **p2))])

svr_model, svr_params = train_evaluate_model(
    'SVR', _svr_objective, _svr_build, X_train_svr, y_train_svr, groups_svr, fold_jobs=N_JOBS)


## XGBoost

In [ ]:
def _xgb_objective(trial):
    p = dict(
        n_estimators=trial.suggest_int('n_estimators', 50, 600),
        learning_rate=trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        max_depth=trial.suggest_int('max_depth', 2, 10),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
    )
    return xgb.XGBRegressor(random_state=0, verbosity=0, tree_method='hist', n_jobs=1, **p)

def _xgb_build(params):
    # single thread: deterministic, and immune to the OpenMP slow-down when the CPU is busy
    return xgb.XGBRegressor(random_state=0, verbosity=0, tree_method='hist', n_jobs=1, **params)

xgb_model, xgb_params = train_evaluate_model(
    'XGBoost', _xgb_objective, _xgb_build, X_train, y_train, groups_train, fold_jobs=N_JOBS)


## LightGBM

In [ ]:
def _lgb_objective(trial):
    p = dict(
        n_estimators=trial.suggest_int('n_estimators', 50, 600),
        learning_rate=trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        num_leaves=trial.suggest_int('num_leaves', 16, 256),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 0.0, 20.0),
    )
    return lgb.LGBMRegressor(random_state=0, verbose=-1, n_jobs=1, **p)

def _lgb_build(params):
    # single thread: deterministic, and immune to the OpenMP slow-down when the CPU is busy
    return lgb.LGBMRegressor(random_state=0, verbose=-1, n_jobs=1, **params)

lgb_model, lgb_params = train_evaluate_model(
    'LightGBM', _lgb_objective, _lgb_build, X_train, y_train, groups_train, fold_jobs=N_JOBS)


## CatBoost

In [ ]:
def _cat_objective(trial):
    p = dict(
        iterations=trial.suggest_int('iterations', 100, 600),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        depth=trial.suggest_int('depth', 3, 10),
        l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 0.5, 15.0, log=True),
    )
    return cat.CatBoostRegressor(random_seed=0, verbose=0, thread_count=N_JOBS, **p)

def _cat_build(params):
    return cat.CatBoostRegressor(random_seed=0, verbose=0, thread_count=N_JOBS, **params)

cat_model, cat_params = train_evaluate_model(
    'CatBoost', _cat_objective, _cat_build, X_train, y_train, groups_train)


## Model Comparison Summary

In [ ]:
evaluation_reg_sorted = evaluation_reg.sort_values('R-squared (test)', ascending=False).reset_index(drop=True)
print(evaluation_reg_sorted.to_string(index=False))

evaluation_reg.to_csv(os.path.join(OUT_DIR, 'metrics.csv'), index=False)

model_order = evaluation_reg['Model'].tolist()
colors = [MODEL_COLORS[m] for m in model_order]
x = np.arange(len(model_order))

fig, axes = plt.subplots(1, 2, figsize=(160/25.4, 90/25.4))
for ax, col, label in [(axes[0], 'R-squared (test)', 'R$^2$ (test, log space)'),
                        (axes[1], 'RMSE (test)', 'RMSE (test, log space)')]:
    vals = evaluation_reg[col].values
    bars = ax.bar(x, vals, color=colors, alpha=0.8, edgecolor='none', width=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(model_order, fontsize=8, rotation=15, ha='right')
    ax.set_ylabel(label, fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', lw=0.4, alpha=0.4)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)
fig.suptitle(f'IDA Stage 1 model comparison (seed={RNG_SEED})', **fonttitle)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig1_r2_rmse.png'), dpi=300, bbox_inches='tight')
plt.show()


## SHAP Comparison


In [ ]:
shap_summary_rows = []
for mn, sv in shap_vals_all.items():
    stem = mn.lower().replace(' ', '_')
    pd.DataFrame(sv, columns=feat_disp).to_csv(os.path.join(SHAP_DIR, f'shap_{stem}_values.csv'), index=False)
    mean_abs = np.abs(sv).mean(axis=0)
    rank = np.argsort(mean_abs)[::-1]
    for r, fi in enumerate(rank, start=1):
        shap_summary_rows.append({'model': mn, 'rank': r, 'feature': feat_disp[fi],
                                   'mean_abs_shap': float(mean_abs[fi])})
pd.DataFrame(shap_summary_rows).to_csv(os.path.join(SHAP_DIR, 'shap_summary.csv'), index=False)

models_done = [m for m in model_order if m in shap_vals_all]
fig_cmp, axes_cmp = plt.subplots(1, len(models_done), figsize=(4.0*len(models_done), 4.5), sharey=True)
if len(models_done) == 1:
    axes_cmp = [axes_cmp]
for ax_, mn in zip(axes_cmp, models_done):
    ma = np.abs(shap_vals_all[mn]).mean(axis=0)
    order = np.argsort(ma)
    ax_.barh([feat_disp[i] for i in order], ma[order], color=MODEL_COLORS[mn], edgecolor='none', height=0.6, alpha=0.85)
    ax_.set_title(mn, fontsize=9, fontweight='bold', color=MODEL_COLORS[mn])
    ax_.set_xlabel('Mean |SHAP|', fontsize=8)
    ax_.spines[['top', 'right']].set_visible(False)
fig_cmp.suptitle(f'Feature Importance — SHAP (full dataset, seed={RNG_SEED})', **fonttitle)
fig_cmp.tight_layout()
fig_cmp.savefig(os.path.join(SHAP_DIR, 'fig_shap_compare.png'), dpi=300, bbox_inches='tight')
plt.show()


## Save Remaining Outputs

In [ ]:
df_uncertainty = evaluation_reg.rename(columns={
    'RMSE (test)': 'RMSE_test', 'R-squared (test)': 'R2_test',
}).copy()
df_uncertainty['Bias_test'] = [
    np.mean(np.array([p['prediction_error'] for p in predictions_all
                       if p['model'] == m and p['set'] == 'test']))
    for m in evaluation_reg['Model']
]
df_uncertainty['Sigma_pre_test'] = [
    np.std(np.array([p['prediction_error'] for p in predictions_all
                      if p['model'] == m and p['set'] == 'test']), ddof=1)
    for m in evaluation_reg['Model']
]
df_uncertainty[['Model', 'Bias_test', 'Sigma_pre_test', 'RMSE_test', 'R2_test']].to_csv(
    os.path.join(OUT_DIR, 'prediction_uncertainty.csv'), index=False)

pd.DataFrame(predictions_all).to_csv(os.path.join(OUT_DIR, 'predictions.csv'), index=False)
pd.DataFrame(trial_rows_all).to_csv(os.path.join(OUT_DIR, 'convergence_trials.csv'), index=False)

best_params_all['_meta'] = dict(
    feat_names=feat_names, feat_disp=feat_disp,
    n_train=int(len(idx_tr)), n_test=int(len(idx_te)),
    n_models_train=int(len(mid_tr)), n_models_test=int(len(mid_te)),
    RNG_SEED=RNG_SEED,
)
with open(os.path.join(OUT_DIR, 'best_params.json'), 'w') as f:
    json.dump(best_params_all, f, indent=2)

df_pred = pd.DataFrame(predictions_all)
fig, axes = plt.subplots(len(model_order), 2, figsize=(180/25.4, (48*len(model_order))/25.4))
for ri, mn in enumerate(model_order):
    for ci, set_name in enumerate(['train', 'test']):
        ax = axes[ri, ci]
        sub = df_pred[(df_pred['model'] == mn) & (df_pred['set'] == set_name)]
        lo = min(sub['y_true'].min(), sub['y_pred'].min()) - 0.1
        hi = max(sub['y_true'].max(), sub['y_pred'].max()) + 0.1
        ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8)
        ax.scatter(sub['y_true'], sub['y_pred'], s=2, alpha=0.15, color=MODEL_COLORS[mn], linewidths=0)
        r2s = r2_score(sub['y_true'], sub['y_pred'])
        rmse = np.sqrt(mean_squared_error(sub['y_true'], sub['y_pred']))
        ax.text(0.05, 0.95, f'R2={r2s:.3f}\nRMSE={rmse:.3f}', transform=ax.transAxes,
                fontsize=7, va='top', bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ccc', lw=0.7))
        if ri == 0:
            ax.set_title('(a) Train set' if set_name == 'train' else '(b) Test set', fontsize=9, fontweight='bold')
        ax.set_xlabel(r'Actual $\ln(\theta_{\max})$', fontsize=7)
        ax.set_ylabel(r'Predicted $\ln(\theta_{\max})$', fontsize=7)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_aspect('equal', adjustable='box')
        ax.spines[['top', 'right']].set_visible(False)
        if ci == 0:
            ax.text(-0.34, 0.5, mn, transform=ax.transAxes, rotation=90, va='center', ha='center',
                    fontsize=8, fontweight='bold', color=MODEL_COLORS[mn])
fig.suptitle(f'Predicted vs. Actual (seed={RNG_SEED}, train and test sets)', **fonttitle)
fig.tight_layout(rect=[0.04, 0, 1, 0.98])
fig.savefig(os.path.join(OUT_DIR, 'fig_scatter.png'), dpi=300, bbox_inches='tight')
plt.show()

print('All outputs ->', OUT_DIR)
